# SDN DDoS Dataset Generator

This notebook generates a **synthetic, ML-ready SDN DDoS flow dataset**. It does **not** launch real DDoS traffic, contact external systems, perform reflection/amplification, or send packets over a network.

The goal is to reproduce the **observable statistical behavior** of several DDoS families so that we can develop and validate the Kafka → PySpark → ML pipeline safely.

## Attack families represented

- BENIGN
- UDP_FLOOD
- ICMP_FLOOD
- UDP_AMPLIFICATION
- SYN_FLOOD
- PING_OF_DEATH_STYLE
- SMURF_STYLE
- HTTP_FLOOD
- SLOWLORIS_STYLE

The generated records represent fixed observation windows, e.g. 5 seconds. Each row is a flow/window observation with network and SDN-oriented features plus a ground-truth label.


## 1. Dataset design

A row represents an observation window rather than an individual packet.

For each observation we generate features such as:

- source/destination information
- protocol and ports
- packet and byte counts
- packet rate and byte rate
- average packet size
- flow duration
- active/new flow counts
- TCP flag behavior
- request rate
- connection completion ratio
- flow entropy
- switch/controller-oriented indicators
- attack label

The important design principle is that the dataset is generated from **behavioral distributions**, not by randomly assigning labels after creating arbitrary rows.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

OUTPUT_DIR = Path("ddos_dataset_output")
OUTPUT_DIR.mkdir(exist_ok=True)

OBS_WINDOW_SECONDS = 5

ATTACKS = [
    "BENIGN",
    "UDP_FLOOD",
    "ICMP_FLOOD",
    "UDP_AMPLIFICATION",
    "SYN_FLOOD",
    "PING_OF_DEATH_STYLE",
    "SMURF_STYLE",
    "HTTP_FLOOD",
    "SLOWLORIS_STYLE",
]

ATTACK_ID = {name: i for i, name in enumerate(ATTACKS)}
ATTACK_ID


## 2. Behavioral model

The generator uses different distributions for each traffic family.

This is intentionally approximate. It is suitable for **pipeline development and controlled experiments**, but it should not be presented as a replacement for packet captures or measurements from a real SDN testbed.

Later, once the Mininet/OVS environment is working, we should create a second dataset from measured flow statistics and compare it with this synthetic dataset.


In [ ]:
def clipped_normal(mean, std, low=0, high=None, size=1):
    x = rng.normal(mean, std, size)
    x = np.maximum(x, low)
    if high is not None:
        x = np.minimum(x, high)
    return x

def lognormal(median, sigma, size=1):
    # Parameterization around an approximate median.
    return rng.lognormal(np.log(median), sigma, size)

def beta_ratio(a, b, size=1):
    return rng.beta(a, b, size)

def choose_protocol(label):
    if label in {"UDP_FLOOD", "UDP_AMPLIFICATION"}:
        return "UDP"
    if label in {"ICMP_FLOOD", "PING_OF_DEATH_STYLE", "SMURF_STYLE"}:
        return "ICMP"
    if label in {"SYN_FLOOD", "HTTP_FLOOD", "SLOWLORIS_STYLE"}:
        return "TCP"
    return rng.choice(["TCP", "UDP", "ICMP"], p=[0.55, 0.30, 0.15])

def choose_ports(label):
    if label == "HTTP_FLOOD":
        return int(rng.integers(32768, 61000)), int(rng.choice([80, 443]))
    if label == "SLOWLORIS_STYLE":
        return int(rng.integers(32768, 61000)), int(rng.choice([80, 443]))
    if label == "SYN_FLOOD":
        return int(rng.integers(32768, 61000)), int(rng.choice([22, 80, 443, 8080]))
    if label in {"UDP_FLOOD", "UDP_AMPLIFICATION"}:
        return int(rng.integers(32768, 61000)), int(rng.integers(1, 65535))
    return 0, 0


In [ ]:
def generate_row(label, timestamp, host_count=12):
    protocol = choose_protocol(label)
    src_id = int(rng.integers(1, host_count + 1))
    dst_id = int(rng.integers(1, host_count + 1))
    while dst_id == src_id:
        dst_id = int(rng.integers(1, host_count + 1))

    src_ip = f"10.0.0.{src_id}"
    dst_ip = f"10.0.0.{dst_id}"
    src_port, dst_port = choose_ports(label)

    # Baseline behavior
    params = {
        "BENIGN": dict(
            packets=clipped_normal(45, 18, 5, 120)[0],
            avg_size=clipped_normal(700, 220, 80, 1500)[0],
            duration=clipped_normal(3.2, 0.9, 0.3, 5)[0],
            new_flows=clipped_normal(5, 2, 1, 12)[0],
            active_flows=clipped_normal(18, 5, 3, 35)[0],
            completion=beta_ratio(8, 2)[0],
            request_rate=clipped_normal(8, 3, 0.2, 20)[0],
            syn=clipped_normal(8, 3, 0, 30)[0],
            ack=clipped_normal(12, 4, 0, 40)[0],
            rst=clipped_normal(1.5, 1, 0, 8)[0],
            entropy=clipped_normal(0.82, 0.08, 0.3, 1.0)[0],
            table_growth=clipped_normal(1.5, 1, 0, 5)[0],
            controller_load=clipped_normal(0.12, 0.04, 0.02, 0.3)[0],
        ),
        "UDP_FLOOD": dict(
            packets=lognormal(1800, 0.45)[0],
            avg_size=clipped_normal(420, 100, 64, 900)[0],
            duration=clipped_normal(4.6, 0.3, 1, 5)[0],
            new_flows=lognormal(90, 0.5)[0],
            active_flows=lognormal(250, 0.45)[0],
            completion=beta_ratio(2, 8)[0],
            request_rate=lognormal(320, 0.45)[0],
            syn=0, ack=0, rst=0,
            entropy=clipped_normal(0.38, 0.12, 0.05, 0.75)[0],
            table_growth=lognormal(45, 0.45)[0],
            controller_load=clipped_normal(0.65, 0.12, 0.2, 1)[0],
        ),
        "ICMP_FLOOD": dict(
            packets=lognormal(2200, 0.4)[0],
            avg_size=clipped_normal(256, 55, 64, 600)[0],
            duration=clipped_normal(4.7, 0.2, 1, 5)[0],
            new_flows=lognormal(65, 0.45)[0],
            active_flows=lognormal(180, 0.4)[0],
            completion=beta_ratio(1.5, 9)[0],
            request_rate=lognormal(380, 0.4)[0],
            syn=0, ack=0, rst=0,
            entropy=clipped_normal(0.30, 0.10, 0.03, 0.7)[0],
            table_growth=lognormal(30, 0.45)[0],
            controller_load=clipped_normal(0.55, 0.12, 0.15, 1)[0],
        ),
        "UDP_AMPLIFICATION": dict(
            packets=lognormal(3500, 0.4)[0],
            avg_size=clipped_normal(950, 220, 150, 1500)[0],
            duration=clipped_normal(4.8, 0.15, 1, 5)[0],
            new_flows=lognormal(35, 0.4)[0],
            active_flows=lognormal(120, 0.4)[0],
            completion=beta_ratio(1.5, 10)[0],
            request_rate=lognormal(260, 0.35)[0],
            syn=0, ack=0, rst=0,
            entropy=clipped_normal(0.52, 0.12, 0.05, 0.9)[0],
            table_growth=lognormal(20, 0.4)[0],
            controller_load=clipped_normal(0.72, 0.10, 0.2, 1)[0],
        ),
        "SYN_FLOOD": dict(
            packets=lognormal(1400, 0.5)[0],
            avg_size=clipped_normal(64, 10, 40, 120)[0],
            duration=clipped_normal(4.9, 0.12, 1, 5)[0],
            new_flows=lognormal(220, 0.5)[0],
            active_flows=lognormal(500, 0.45)[0],
            completion=beta_ratio(1, 20)[0],
            request_rate=lognormal(300, 0.45)[0],
            syn=lognormal(900, 0.4)[0],
            ack=clipped_normal(8, 5, 0, 30)[0],
            rst=lognormal(55, 0.5)[0],
            entropy=clipped_normal(0.42, 0.12, 0.05, 0.8)[0],
            table_growth=lognormal(100, 0.45)[0],
            controller_load=clipped_normal(0.85, 0.08, 0.3, 1)[0],
        ),
        "PING_OF_DEATH_STYLE": dict(
            packets=lognormal(900, 0.45)[0],
            avg_size=clipped_normal(1450, 60, 1200, 1500)[0],
            duration=clipped_normal(4.5, 0.3, 1, 5)[0],
            new_flows=lognormal(20, 0.4)[0],
            active_flows=lognormal(80, 0.35)[0],
            completion=beta_ratio(2, 7)[0],
            request_rate=lognormal(140, 0.4)[0],
            syn=0, ack=0, rst=0,
            entropy=clipped_normal(0.55, 0.12, 0.05, 0.9)[0],
            table_growth=lognormal(12, 0.35)[0],
            controller_load=clipped_normal(0.42, 0.10, 0.1, 1)[0],
        ),
        "SMURF_STYLE": dict(
            packets=lognormal(2600, 0.5)[0],
            avg_size=clipped_normal(500, 130, 80, 1200)[0],
            duration=clipped_normal(4.7, 0.2, 1, 5)[0],
            new_flows=lognormal(100, 0.5)[0],
            active_flows=lognormal(300, 0.45)[0],
            completion=beta_ratio(1.5, 9)[0],
            request_rate=lognormal(450, 0.45)[0],
            syn=0, ack=0, rst=0,
            entropy=clipped_normal(0.25, 0.10, 0.02, 0.65)[0],
            table_growth=lognormal(40, 0.4)[0],
            controller_load=clipped_normal(0.68, 0.12, 0.2, 1)[0],
        ),
        "HTTP_FLOOD": dict(
            packets=lognormal(700, 0.45)[0],
            avg_size=clipped_normal(850, 220, 150, 1500)[0],
            duration=clipped_normal(4.2, 0.6, 0.5, 5)[0],
            new_flows=lognormal(80, 0.45)[0],
            active_flows=lognormal(220, 0.4)[0],
            completion=beta_ratio(7, 3)[0],
            request_rate=lognormal(180, 0.4)[0],
            syn=lognormal(120, 0.4)[0],
            ack=lognormal(180, 0.4)[0],
            rst=clipped_normal(10, 5, 0, 40)[0],
            entropy=clipped_normal(0.68, 0.10, 0.1, 1)[0],
            table_growth=lognormal(55, 0.4)[0],
            controller_load=clipped_normal(0.62, 0.12, 0.15, 1)[0],
        ),
        "SLOWLORIS_STYLE": dict(
            packets=clipped_normal(25, 10, 5, 70)[0],
            avg_size=clipped_normal(120, 30, 40, 300)[0],
            duration=clipped_normal(4.9, 0.12, 2, 5)[0],
            new_flows=lognormal(130, 0.4)[0],
            active_flows=lognormal(420, 0.4)[0],
            completion=beta_ratio(1, 12)[0],
            request_rate=clipped_normal(35, 12, 2, 80)[0],
            syn=lognormal(100, 0.4)[0],
            ack=clipped_normal(18, 8, 0, 50)[0],
            rst=clipped_normal(2, 2, 0, 15)[0],
            entropy=clipped_normal(0.72, 0.10, 0.1, 1)[0],
            table_growth=lognormal(80, 0.4)[0],
            controller_load=clipped_normal(0.78, 0.10, 0.2, 1)[0],
        ),
    }[label]

    packets = max(1, int(params["packets"]))
    avg_size = float(params["avg_size"])
    byte_count = max(packets * avg_size, packets)
    packet_rate = packets / OBS_WINDOW_SECONDS
    byte_rate = byte_count / OBS_WINDOW_SECONDS

    # Mild correlation: higher packet volumes tend to create more flow pressure.
    flow_rate = max(0.1, params["new_flows"] / OBS_WINDOW_SECONDS)
    active_flows = max(1, int(params["active_flows"]))

    return {
        "timestamp": timestamp,
        "observation_window_s": OBS_WINDOW_SECONDS,
        "switch_id": f"s{int(rng.integers(1, 4))}",
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "src_port": src_port,
        "dst_port": dst_port,
        "protocol": protocol,
        "packet_count": packets,
        "byte_count": round(byte_count, 2),
        "flow_duration_s": round(float(params["duration"]), 4),
        "packet_rate": round(packet_rate, 4),
        "byte_rate": round(byte_rate, 4),
        "avg_packet_size": round(avg_size, 4),
        "new_flows": int(max(1, params["new_flows"])),
        "active_flows": active_flows,
        "flow_rate": round(flow_rate, 4),
        "tcp_syn_count": int(max(0, params["syn"])),
        "tcp_ack_count": int(max(0, params["ack"])),
        "tcp_rst_count": int(max(0, params["rst"])),
        "tcp_connection_completion_ratio": round(float(params["completion"]), 5),
        "request_rate": round(float(params["request_rate"]), 4),
        "source_entropy": round(float(params["entropy"]), 5),
        "flow_table_growth": round(float(params["table_growth"]), 4),
        "controller_load_indicator": round(float(params["controller_load"]), 5),
        "label": label,
        "label_id": ATTACK_ID[label],
    }


## 3. Generate the dataset

The default dataset contains 2,000 observations per class.

For an initial ML experiment this is enough to validate the pipeline. Later we can increase the sample count and, more importantly, introduce scenario variation and noise.


In [ ]:
N_PER_CLASS = 2000
start_time = pd.Timestamp("2026-01-01 00:00:00", tz="UTC")

rows = []
for label in ATTACKS:
    for i in range(N_PER_CLASS):
        ts = start_time + pd.Timedelta(seconds=(len(rows) * OBS_WINDOW_SECONDS))
        rows.append(generate_row(label, ts))

df = pd.DataFrame(rows)

# Shuffle only after generation so class-specific generation order is not preserved.
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts())
df.head()


## 4. Sanity checks

Before using the dataset, inspect whether the generated classes actually differ in meaningful ways.

Do not jump directly to model training. First inspect the distributions.


In [ ]:
summary = (
    df.groupby("label")
      .agg(
          packet_rate_mean=("packet_rate", "mean"),
          byte_rate_mean=("byte_rate", "mean"),
          avg_packet_size_mean=("avg_packet_size", "mean"),
          new_flows_mean=("new_flows", "mean"),
          active_flows_mean=("active_flows", "mean"),
          completion_ratio_mean=("tcp_connection_completion_ratio", "mean"),
          controller_load_mean=("controller_load_indicator", "mean"),
      )
      .round(2)
)

summary


In [ ]:
import matplotlib.pyplot as plt

features_to_plot = [
    "packet_rate",
    "byte_rate",
    "new_flows",
    "active_flows",
    "tcp_connection_completion_ratio",
    "controller_load_indicator",
]

for feature in features_to_plot:
    plt.figure(figsize=(11, 5))
    df.boxplot(column=feature, by="label", rot=45)
    plt.title(feature)
    plt.suptitle("")
    plt.tight_layout()
    plt.show()


## 5. Correlation inspection

A useful next check is whether some features are almost duplicates.

For example:

- packet_count and packet_rate are mathematically related because the observation window is fixed.
- byte_count and byte_rate are similarly related.

For the final ML dataset we may retain both for interpretability, but we should understand this relationship before training.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Numeric Feature Correlation")
plt.tight_layout()
plt.show()


## 6. Export

We export both CSV and Parquet.

Parquet is preferable for the later Spark pipeline because it is columnar and works naturally with distributed analytics.


In [ ]:
csv_path = OUTPUT_DIR / "sdn_ddos_synthetic.csv"
parquet_path = OUTPUT_DIR / "sdn_ddos_synthetic.parquet"

df.to_csv(csv_path, index=False)

try:
    df.to_parquet(parquet_path, index=False)
    parquet_status = str(parquet_path)
except Exception as e:
    parquet_status = f"Parquet export unavailable: {e}"

print("CSV:", csv_path)
print("Parquet:", parquet_status)


## 7. Create a train/test split without leakage

For network data, a random row split can be misleading because nearby observations from the same scenario can end up in both train and test sets.

For this synthetic first version, we create a simple deterministic split by row order after sorting by timestamp. In the next version we should add an explicit `scenario_id` and split by scenario, not by individual row.

This is important because the final evaluation should answer:

> Can the model detect a previously unseen traffic scenario?

rather than:

> Can the model recognize rows generated by the same scenario it already saw?


In [ ]:
df_time = df.sort_values("timestamp").reset_index(drop=True)

split = int(len(df_time) * 0.8)
train_df = df_time.iloc[:split].copy()
test_df = df_time.iloc[split:].copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts(normalize=True).round(3))

print("\nTest labels:")
print(test_df["label"].value_counts(normalize=True).round(3))


## 8. What this dataset is — and is not

### It is

A controlled synthetic dataset for:

- developing the SDN DDoS detection pipeline
- testing feature engineering
- testing Kafka ingestion
- testing PySpark Structured Streaming
- testing Spark ML models
- building initial visualizations
- debugging the end-to-end architecture

### It is not

It is not evidence that a real network produces exactly these distributions.

For the final project, we should build a second-generation dataset using:

**Mininet → OVS → controller → controlled laboratory traffic → measured flow statistics**

and then compare the measured distributions with this synthetic dataset.

The strongest final methodology is therefore:

1. Synthetic dataset for rapid pipeline development.
2. Controlled SDN-lab measurements for experimental validation.
3. Optional public benchmark dataset for external comparison.


## 9. Next extension

The next version of this notebook should add:

- `scenario_id`
- `attacker_count`
- `attack_intensity`
- `background_traffic_level`
- `destination_service`
- `switch_port`
- `in_packets`
- `out_packets`
- `in_bytes`
- `out_bytes`
- `flow_churn`
- 5 s / 10 s / 30 s aggregation windows
- noisy and borderline cases
- class imbalance
- scenario-level train/test splitting

Then we can feed the generated records into the same Kafka schema that the real SDN collector will eventually use.
